# Paso 1: Abrimos el archivo de datos

In [2]:
# colocamos todas las librerías a emplear en el presente proyecto
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [3]:
# leemos el dataset y lo colocamos en un dataframe
df = pd.read_csv('../datasets/users_behavior.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


se verifica que no existen valores nulos en cada una de las columnas o atributos del dataframe. Se aprecia además que existen tipos de valores float en los atributos 'calls' y 'messages', deberían ser tipo int.

In [4]:
# Comprobamos la variable calls y messages para convertir de float a int.
convert_to_int_calls=np.array_equal(df['calls'], df['calls'].astype('int'))
convert_to_int_messages=np.array_equal(df['messages'], df['messages'].astype('int'))
print(convert_to_int_calls, convert_to_int_messages)

True True


se verificó que las columnas pueden cambiarse a un tipo int.

In [5]:
# se cambia los tipos de datos de float a int en las columnas mencionadas
df['calls'] = df['calls'].astype('int')
df['messages'] = df['messages'].astype('int')

los datos se encuentran limpios para usarse en el análisis

# Paso 2. Se segmenta los datos fuente en un conjunto de entrenamiento, uno de validación y uno de prueba.

In [6]:
# extraemos la columna is_ultra porque es el target u objetivo. Además, se coloca el parámetro axis para seleccionar columna.
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

In [7]:
# se segmenta la data en los conjuntos entrenamiento y prueba. Se emplea un 20% para el dataframe de prueba.
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.20, random_state=14)

# se segmenta la data en los conjuntos entrenamiento y validación. Se emplea un test_size=0.25 para obtener el 20% del df original.
features_train, features_val, target_train, target_val = train_test_split(
    features_train, target_train, test_size=0.25, random_state=14)

In [8]:
# se comprueba que el df_train es el 60% del df original
train_percent=features_train.shape[0]/df.shape[0]
# se comprueba que el df_valid es el 20% del df original
val_percent=features_val.shape[0]/df.shape[0]
# se comprueba que el df_test es el 20% del df original
test_percent=features_test.shape[0]/df.shape[0]
print(train_percent*100,val_percent*100,test_percent*100)

59.98755444928439 20.00622277535781 20.00622277535781


se comprueban los datos del 60% para los entrenamientos, 20% para las pruebas y 20% para las validaciones

# Paso 3. Se investiga la calidad de diferentes modelos cambiando los hiperparámetros.

### Desarrolla un modelo con la mayor exactitud posible. En este proyecto, el umbral de exactitud es 0.75. Usa el dataset para comprobar la exactitud.

- El objetivo es clasificar a los nuevos usuarios en alguno de los planes: Smart o Ultra. También, se quiere que la exactitud de los datos del modelo sea la mayor posible.
- Se analizan los modelos para clasificar como los el árbol de decisión y bosques aleatorios.
- Se analiza el método GridSearchCV generando los mejores parámetros para el modelo de arbol de decisión y para el bosque aleatorio.

In [9]:
# Definimos los parámetros grid para la búsqueda
param_grid = {'max_depth': range(1, 21), 'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 5, 10], 'criterion': ['gini', 'entropy']}

# Creamos el modelo base de árbol de decisiones
model = DecisionTreeClassifier(
    random_state=14,
    min_samples_split=10,
    min_samples_leaf=5
)

# Instanciamos el modelo de búsqueda grid con validación cruzada de 5 pasos
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1  # Usamos todos los núcleos disponibles
)

# Se realiza la búsqueda con el conjunto de entrenamiento.
grid_search.fit(features_train, target_train)

# Se obtienen los mejores parámetros y la mejor puntuación
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Se evalúa y valida
best_model = grid_search.best_estimator_
val_score = best_model.score(features_val, target_val)

print(f"Best parameters: {best_params}")
print(f"Best cross-validation accuracy: {best_score:.4f}")
print(f"Validation set accuracy: {val_score:.4f}")

Best parameters: {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 5, 'min_samples_split': 20}
Best cross-validation accuracy: 0.8008
Validation set accuracy: 0.8009


- Se han obtenido los mejores parámetros para el modelo. El criterio es de ganancia de información para decidir como se separa la data. La profundidad máxima del árbol es de 7, Esto previene un sobreajuste del modelo.
- Best cross-validation accuracy: 0.8008, este valor indica que existe una validación cruzada para una exactitud promedio de 80.08%. Esto estima como el modelo se va a comportar con data no vista.
- Validation set accuracy: 0.8009, este valor de exactitud ha usado data de validación, el cual ha alcanzado un valor de 80.09%, lo cual es muy cercano a la validación cruzada. Esta aproximación nos indica que el modelo no está sobreajustado y que tiene un buen comportamiento.

In [11]:
# Definimos los parámetros grid para la búsqueda
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.8],
    'criterion': ['gini', 'entropy']
}
# Creamos el modelo base de bosque aleatorio
model = RandomForestClassifier(
    random_state=15,
    min_samples_split=10,
    min_samples_leaf=5,
    bootstrap=True,
    oob_score=True  # Se habilitado la evaluación out-of-bag
)

# Se configura GridSearchCV con validación cruzada estratificada de 5 particiones
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

# Se realiza la búsqueda con el conjunto de entrenamiento.
grid_search.fit(features_train, target_train)

# Se obtienen los mejores parámetros y la mejor puntuación
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Se evalúa y valida
best_model = grid_search.best_estimator_
val_score = best_model.score(features_val, target_val)

print(f"Best parameters: {best_params}")
print(f"Best cross-validation accuracy: {best_score:.4f}")
print(f"Out-of-bag accuracy: {best_model.oob_score_:.4f}")
print(f"Validation set accuracy: {val_score:.4f}")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best parameters: {'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best cross-validation accuracy: 0.8102
Out-of-bag accuracy: 0.8055
Validation set accuracy: 0.8025


- Los bosques no pueden sobreajustarse debido a que tienen demasiados árboles. Si bien el sobreajuste de un bosque aún puede ocurrir este efecto generalmente se ve compensado por el beneficio de tener muchos árboles.
- Best cross-validation accuracy: 0.8102 y Validation set accuracy: 0.8025 presentan una diferencia mínima que indica que no se existirá sobreajuste en el modelo.
- No se limita la profundidad de los árboles del bosque
- En general es un buen modelo porque todas las exactitudes bridan valores mayores a 80%
- Se evita el sobreajuste del modelo con la corta distancia entre Best cross-validation accuracy y Validation set accuracy.
- Con la cantidad de 100 estimadores se considera adecuado. Valor igual al estandar.
- La cercanía de los tres tipos de exactitudes mencionadas indica que el modelo tendrá un comportamiento confiable con nueva data

# Paso 4. Se comprueba la calidad del modelo usando el conjunto de prueba.

### Se entrena cada modelo para obtener la cantidad de errores que genera y la exactitud que presenta.

In [12]:
# Empleamos el modelo de arbol de decisiones en el conjunto de entrenamiento y con los parámetros para un mejor ajuste del modelo
model_tree = DecisionTreeClassifier(criterion= 'entropy', max_depth= 7, min_samples_leaf= 5, min_samples_split= 20)
model_tree.fit(features_train, target_train)
# Predecimos los valores target para los datos de prueba (test)
test_predictions = model_tree.predict(features_test)
# Calculamos la métrica de exactitud en el conjunto de prueba
accuracy = accuracy_score(target_test, test_predictions)
print(f'Exactitud: {accuracy:.4f}')

Exactitud: 0.8056


In [13]:
# Empleamos el modelo de bosque aleatorios en el conjunto de entrenamiento y con los parámetros para un mejor ajuste del modelo
model_forest = RandomForestClassifier(criterion= 'gini', max_depth= None, max_features= 'sqrt', min_samples_split= 2, n_estimators= 100)
model_forest.fit(features_train, target_train)
# Predecimos los valores target para los datos de prueba (test)
test_predictions = model_forest.predict(features_test)
# Calculamos la métrica de exactitud en el conjunto de prueba
accuracy = accuracy_score(target_test, test_predictions)
print(f'Exactitud: {accuracy:.4f}')

Exactitud: 0.8212


- El modelo seleccionado es el RandomForestClassifier por presentar una exactitud más elevada entre los modelos entrenados.
- Se cumple con desarrollar un modelo con la mayor exactitud posible sobrepasando el umbral establecido en el problema de 75% alcanzando un valor de 82.12%.